# NeuroXAI: Colab-Ready Advanced Notebook (MRI + SHAP)

This notebook provides a comprehensive, GPU-enabled pipeline for Alzheimer’s MRI classification with extensive A–Z visualizations and explainability (SHAP, Grad-CAM, Integrated Gradients, Occlusion), designed to run on Google Colab using real data.

- Supports: local folders, Kaggle downloads, or Google Drive datasets
- Outputs: HD plots, dashboards, and exportable reports
- Classes: Non-Demented, Very Mild, Mild, Moderate

Follow each section in order. Configure your dataset path below (real MRI images only).


In [ ]:
# Environment setup: GPU + dependencies (Colab-friendly)
import os, sys, subprocess, platform
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print('GPU memory growth not set:', e)
    print('GPUs available:', gpus)
except Exception as e:
    print('TensorFlow import failed:', e)
    raise

def pip_install_pkgs(pkgs):
    for p in pkgs:
        print('Installing', p)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', p])
        
required = [
    'tensorflow>=2.14',
    'numpy', 'pandas', 'matplotlib', 'seaborn',
    'scikit-learn', 'scikit-image', 'opencv-python',
    'plotly', 'umap-learn', 'shap', 'nibabel'
]
pip_install_pkgs(required)
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
import cv2, shap, plotly.express as px, plotly.graph_objects as go
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from skimage import filters, exposure, measure
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import plot_model
print('Environment ready. TF version:', tf.__version__)

In [ ]:
# ====== DATASET CONFIGURATION (MATCHES YOUR EXACT FOLDER STRUCTURE) ======
# Your data location: Datasets/MRI Dataset/train.parquet and test.parquet
# ALZ Variant Dataset: Datasets/ALZ_Variant Datset/preprocessed_alz_data.npz

from google.colab import drive as colab_drive
import io
from PIL import Image

# ===== CONFIGURATION =====
# For Google Colab: set COLAB_DATA_PATH after mounting drive
# For Local VS Code: paths work directly

# Class names matching your exact dataset
CLASSES = ['Non-Demented', 'Very Mild Dementia', 'Mild Dementia', 'Moderate Dementia']
CLASS_LABELS = {0: 'Non-Demented', 1: 'Very Mild Dementia', 2: 'Mild Dementia', 3: 'Moderate Dementia'}
IMG_SIZE = (128, 128)
SEED = 42

# ===== DATA PATHS (YOUR EXACT STRUCTURE) =====
# MRI Dataset paths (Parquet format)
MRI_TRAIN_PATH = 'Datasets/MRI Dataset/train.parquet'
MRI_TEST_PATH = 'Datasets/MRI Dataset/test.parquet'

# ALZ Variant Dataset path (NPZ format)  
ALZ_VARIANT_PATH = 'Datasets/ALZ_Variant Datset/preprocessed_alz_data.npz'

# Choose which dataset to use: 'mri' or 'alz_variant'
DATASET_TO_USE = 'mri'

def mount_colab_drive():
    """Mount Google Drive for Colab"""
    if IN_COLAB:
        try:
            colab_drive.mount('/content/drive')
            print('✅ Google Drive mounted at /content/drive')
            print('   Update paths below to point to your Drive folder')
            return True
        except Exception as e:
            print(f'⚠️ Drive mount skipped: {e}')
            return False
    return False

def load_mri_parquet_data(train_path, test_path):
    """
    Load MRI data from your Parquet files
    Matches your exact format: train.parquet and test.parquet
    with 'image' (bytes) and 'label' (int) columns
    """
    print(f"📂 Loading MRI data from Parquet files...")
    print(f"   Train: {train_path}")
    print(f"   Test: {test_path}")
    
    def decode_parquet(path):
        df = pd.read_parquet(path)
        images = []
        labels = []
        
        for idx, row in df.iterrows():
            try:
                # Decode image bytes to PIL Image
                img_bytes = row['image']['bytes'] if isinstance(row['image'], dict) else row['image']
                bio = io.BytesIO(img_bytes)
                img = Image.open(bio).convert('L')  # Grayscale
                img = img.resize(IMG_SIZE)
                images.append(np.array(img))
                labels.append(row['label'])
            except Exception as e:
                if idx < 3:
                    print(f"   ⚠️ Decode error at row {idx}: {e}")
        
        return np.array(images), np.array(labels)
    
    X_train, y_train = decode_parquet(train_path)
    X_test, y_test = decode_parquet(test_path)
    
    print(f"✅ Loaded {len(X_train)} train images, {len(X_test)} test images")
    print(f"   Shape: {X_train.shape}, Labels: {np.unique(y_train)}")
    
    return X_train, y_train, X_test, y_test

def load_alz_variant_data(npz_path):
    """
    Load ALZ Variant Dataset from your preprocessed NPZ file
    Expected keys: X_train, X_test, y_train, y_test
    """
    print(f"📂 Loading ALZ Variant data from NPZ file...")
    print(f"   Path: {npz_path}")
    
    data = np.load(npz_path)
    X_train = data['X_train']
    X_test = data['X_test']
    y_train = data['y_train']
    y_test = data['y_test']
    
    print(f"✅ Loaded {len(X_train)} train samples, {len(X_test)} test samples")
    print(f"   Shape: {X_train.shape}")
    
    return X_train, y_train, X_test, y_test

# ===== LOAD YOUR DATA =====
if IN_COLAB:
    mount_colab_drive()
    # Update these paths if your data is in Google Drive:
    # MRI_TRAIN_PATH = '/content/drive/MyDrive/YourFolder/train.parquet'
    # MRI_TEST_PATH = '/content/drive/MyDrive/YourFolder/test.parquet'

print(f"\n🔬 Using dataset: {DATASET_TO_USE.upper()}")
print("=" * 50)

try:
    if DATASET_TO_USE == 'mri':
        X_train_raw, y_train_raw, X_test_raw, y_test_raw = load_mri_parquet_data(MRI_TRAIN_PATH, MRI_TEST_PATH)
    else:
        X_train_raw, y_train_raw, X_test_raw, y_test_raw = load_alz_variant_data(ALZ_VARIANT_PATH)
    
    DATA_LOADED = True
except FileNotFoundError as e:
    print(f"❌ Data not found: {e}")
    print("   Please check your file paths or upload data to Colab")
    DATA_LOADED = False
except Exception as e:
    print(f"❌ Error loading data: {e}")
    DATA_LOADED = False

In [ ]:
# ====== COLAB DATA UPLOAD HELPER (if files not on Drive) ======
from google.colab import files as colab_files

def upload_parquet_to_colab():
    """Interactive upload for Colab users without Drive"""
    if IN_COLAB:
        print("📤 Upload your train.parquet and test.parquet files:")
        uploaded = colab_files.upload()
        print(f"✅ Uploaded: {list(uploaded.keys())}")
        return list(uploaded.keys())
    return []

def upload_npz_to_colab():
    """Interactive upload for NPZ file"""
    if IN_COLAB:
        print("📤 Upload your preprocessed_alz_data.npz file:")
        uploaded = colab_files.upload()
        print(f"✅ Uploaded: {list(uploaded.keys())}")
        return list(uploaded.keys())
    return []

# Uncomment to use interactive upload in Colab:
# upload_parquet_to_colab()
# Then update paths:
# MRI_TRAIN_PATH = 'train.parquet'
# MRI_TEST_PATH = 'test.parquet'

In [ ]:
# ====== PREPROCESSING, SKULL-STRIPPING & DATA AUGMENTATION ======
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def skull_strip_simple(img):
    """Simple threshold-based skull stripping approximation"""
    thr = filters.threshold_otsu(img)
    mask = img > thr
    img2 = img.copy()
    if mask.sum() > 0:
        img2[~mask] = img2[mask].mean()
    return img2

def preprocess_image(img):
    """Full preprocessing pipeline for MRI images"""
    # Resize if needed
    if img.shape != IMG_SIZE:
        img = cv2.resize(img, IMG_SIZE)
    
    # Histogram equalization
    img = exposure.equalize_hist(img)
    img = (img * 255).astype(np.uint8)
    
    # Skull stripping
    img = skull_strip_simple(img)
    
    # Normalize to [0, 1]
    img = img.astype(np.float32) / 255.0
    
    # Add channel dimension (grayscale)
    img = np.expand_dims(img, -1)
    return img

def prepare_data(X_train_raw, y_train_raw, X_test_raw, y_test_raw):
    """Preprocess and split data"""
    print("🔄 Preprocessing images...")
    
    # Preprocess all images
    X_train = np.array([preprocess_image(img) for img in X_train_raw])
    X_test = np.array([preprocess_image(img) for img in X_test_raw])
    
    # Encode labels
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train_raw)
    y_test_encoded = le.transform(y_test_raw)
    
    # One-hot encode
    num_classes = len(le.classes_)
    y_train_oh = keras.utils.to_categorical(y_train_encoded, num_classes)
    y_test_oh = keras.utils.to_categorical(y_test_encoded, num_classes)
    
    # Split train into train/val
    X_train_final, X_val, y_train_final, y_val = train_test_split(
        X_train, y_train_oh, test_size=0.2, random_state=SEED, stratify=y_train_encoded
    )
    
    print(f"✅ Data preprocessed!")
    print(f"   Train: {X_train_final.shape}")
    print(f"   Val:   {X_val.shape}")
    print(f"   Test:  {X_test.shape}")
    print(f"   Classes: {le.classes_}")
    
    return X_train_final, X_val, X_test, y_train_final, y_val, y_test_oh, le, num_classes

# ===== PREPARE YOUR DATA =====
if DATA_LOADED:
    X_train, X_val, X_test, y_train, y_val, y_test, le, num_classes = prepare_data(
        X_train_raw, y_train_raw, X_test_raw, y_test_raw
    )
    
    # Keep raw data for visualizations
    y_train_idx = np.argmax(y_train, axis=1)
    y_val_idx = np.argmax(y_val, axis=1)
    y_test_idx = np.argmax(y_test, axis=1)
else:
    print("⚠️ Data not loaded. Fix paths above and re-run.")

# ===== DATA AUGMENTATION PIPELINE =====
AUTOTUNE = tf.data.AUTOTUNE

def augment(x):
    """Data augmentation for training"""
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.1)
    x = tf.image.random_contrast(x, 0.9, 1.1)
    x = tf.image.rot90(x, k=tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32))
    return x

if DATA_LOADED:
    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))\
        .shuffle(8192, SEED)\
        .map(lambda a, b: (augment(a), b), num_parallel_calls=AUTOTUNE)\
        .batch(64)\
        .prefetch(AUTOTUNE)
    
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))\
        .batch(64)\
        .prefetch(AUTOTUNE)
    
    test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))\
        .batch(64)\
        .prefetch(AUTOTUNE)
    
    print(f"\n✅ TF Datasets ready!")
    print(f"   Batch size: 64, Augmentation: ON")

## 1. Exploratory Data Analysis – Class Distribution

Bar charts, pie charts, and donut charts showing the distribution of dementia stages.

In [ ]:
# ====== EDA: CLASS DISTRIBUTION VISUALIZATIONS ======
from collections import Counter

if DATA_LOADED:
    # Get label counts from training data
    label_counts = Counter(y_train_idx)
    class_names = [CLASS_LABELS.get(i, f'Class {i}') for i in sorted(label_counts.keys())]
    counts = [label_counts[i] for i in sorted(label_counts.keys())]
    df_counts = pd.DataFrame({'Class': class_names, 'Count': counts})

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Bar chart
    colors = sns.color_palette('viridis', len(df_counts))
    bars = axes[0].bar(df_counts['Class'], df_counts['Count'], color=colors, edgecolor='black', linewidth=1.5)
    axes[0].set_title('🧠 Class Distribution (Bar)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Dementia Stage', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].tick_params(axis='x', rotation=15)
    for bar, v in zip(bars, df_counts['Count']):
        axes[0].text(bar.get_x() + bar.get_width()/2, v + 10, str(v), ha='center', fontweight='bold', fontsize=11)

    # Pie chart
    axes[1].pie(df_counts['Count'], labels=df_counts['Class'], autopct='%1.1f%%', colors=colors, 
                startangle=90, explode=[0.03]*len(df_counts), shadow=True)
    axes[1].set_title('🧠 Class Distribution (Pie)', fontsize=14, fontweight='bold')

    # Donut chart
    wedges, texts, autotexts = axes[2].pie(df_counts['Count'], labels=df_counts['Class'], 
                                            autopct='%1.1f%%', colors=colors, startangle=90, pctdistance=0.75)
    centre_circle = plt.Circle((0, 0), 0.5, fc='white')
    axes[2].add_patch(centre_circle)
    axes[2].set_title('🧠 Class Distribution (Donut)', fontsize=14, fontweight='bold')
    
    plt.suptitle('Alzheimer\'s MRI Dataset - Class Distribution', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('class_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n📊 Class Distribution Summary:")
    for cls, cnt in zip(class_names, counts):
        print(f"   {cls}: {cnt} samples ({cnt/sum(counts)*100:.1f}%)")
else:
    print("⚠️ Load data first!")

## 2. Exploratory Data Analysis – Image Statistics

Mean, std, min, max pixel values per class with grouped bar charts and summary tables.

In [ ]:
# ====== EDA: IMAGE STATISTICS PER CLASS ======
if DATA_LOADED:
    stats_list = []
    for cls_idx in range(num_classes):
        cls_name = CLASS_LABELS.get(cls_idx, f'Class {cls_idx}')
        # Get images for this class
        mask = y_train_idx == cls_idx
        imgs = X_train[mask]
        
        if len(imgs) > 0:
            stats_list.append({
                'Class': cls_name,
                'Samples': len(imgs),
                'Mean': f"{imgs.mean():.4f}",
                'Std': f"{imgs.std():.4f}",
                'Min': f"{imgs.min():.4f}",
                'Max': f"{imgs.max():.4f}",
                'Median': f"{np.median(imgs):.4f}"
            })
    
    df_stats = pd.DataFrame(stats_list)
    
    # Display styled table
    print("📊 Image Statistics per Dementia Stage:")
    display(df_stats.style.background_gradient(cmap='Blues', subset=['Samples'])
            .set_properties(**{'text-align': 'center'})
            .set_caption('MRI Image Statistics'))

    # Grouped bar chart for statistics
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(df_stats))
    width = 0.2
    
    means = [float(m) for m in df_stats['Mean']]
    stds = [float(s) for s in df_stats['Std']]
    medians = [float(m) for m in df_stats['Median']]
    
    bars1 = ax.bar(x - width, means, width, label='Mean', color='steelblue', edgecolor='black')
    bars2 = ax.bar(x, stds, width, label='Std', color='coral', edgecolor='black')
    bars3 = ax.bar(x + width, medians, width, label='Median', color='seagreen', edgecolor='black')
    
    ax.set_xlabel('Dementia Stage', fontsize=12)
    ax.set_ylabel('Pixel Value (Normalized)', fontsize=12)
    ax.set_title('🔬 Image Statistics by Class', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(df_stats['Class'], rotation=15)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('image_statistics.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ Load data first!")

## 3. Sample MRI Images Grid by Class

Display sample MRI images for each dementia stage with annotations.

In [ ]:
# ====== SAMPLE MRI IMAGES GRID BY CLASS ======
if DATA_LOADED:
    n_samples = 5
    fig, axes = plt.subplots(num_classes, n_samples, figsize=(15, 3*num_classes))
    
    for row in range(num_classes):
        cls_name = CLASS_LABELS.get(row, f'Class {row}')
        # Get indices for this class
        idx = np.where(y_train_idx == row)[0]
        
        if len(idx) > 0:
            samples = np.random.choice(idx, min(n_samples, len(idx)), replace=False)
            for col, s in enumerate(samples):
                axes[row, col].imshow(X_train[s].squeeze(), cmap='gray')
                axes[row, col].axis('off')
                if col == 0:
                    axes[row, col].set_title(f'Sample {col+1}', fontsize=10)
            
            # Add class label on the left
            axes[row, 0].annotate(cls_name, xy=(-0.15, 0.5), xycoords='axes fraction', 
                                  fontsize=11, fontweight='bold', va='center', rotation=90,
                                  bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    
    fig.suptitle('🧠 Sample MRI Scans per Dementia Stage', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('sample_mri_grid.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Showing {n_samples} samples from each of {num_classes} classes")
else:
    print("⚠️ Load data first!")

## 4. Pixel Intensity Distribution Analysis

Histograms, KDE plots, and cumulative distribution functions across dementia stages.

In [ ]:
# ====== PIXEL INTENSITY DISTRIBUTION ANALYSIS ======
if DATA_LOADED:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    palette = sns.color_palette('Set2', num_classes)
    
    # Histogram
    for i in range(num_classes):
        cls_name = CLASS_LABELS.get(i, f'Class {i}')
        idx = np.where(y_train_idx == i)[0]
        if len(idx) > 0:
            axes[0].hist(X_train[idx].flatten(), bins=50, alpha=0.5, label=cls_name, color=palette[i])
    axes[0].set_title('🔬 Pixel Intensity Histogram', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Pixel Value (Normalized)')
    axes[0].set_ylabel('Frequency')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # KDE
    for i in range(num_classes):
        cls_name = CLASS_LABELS.get(i, f'Class {i}')
        idx = np.where(y_train_idx == i)[0]
        if len(idx) > 0:
            # Sample to avoid memory issues
            sample_data = X_train[idx][:500].flatten()
            sns.kdeplot(sample_data, ax=axes[1], label=cls_name, color=palette[i], fill=True, alpha=0.3)
    axes[1].set_title('🔬 Pixel Intensity KDE', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Pixel Value (Normalized)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    # CDF (Cumulative Distribution)
    for i in range(num_classes):
        cls_name = CLASS_LABELS.get(i, f'Class {i}')
        idx = np.where(y_train_idx == i)[0]
        if len(idx) > 0:
            vals = np.sort(X_train[idx].flatten())
            step = max(1, len(vals) // 1000)  # Subsample for plotting
            cdf = np.arange(1, len(vals)+1) / len(vals)
            axes[2].plot(vals[::step], cdf[::step], label=cls_name, color=palette[i], linewidth=2)
    axes[2].set_title('🔬 Pixel Intensity CDF', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Pixel Value (Normalized)')
    axes[2].set_ylabel('Cumulative Probability')
    axes[2].legend()
    axes[2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('pixel_intensity_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ Load data first!")

## 5. Image Quality Metrics

Contrast, entropy, sharpness, and signal-to-noise ratio per class.

In [ ]:
# ====== IMAGE QUALITY METRICS PER CLASS ======
from skimage.measure import shannon_entropy
from scipy.ndimage import laplace

def compute_quality_metrics(img):
    """Compute image quality metrics"""
    img2d = img.squeeze()
    contrast = img2d.std()
    entropy = shannon_entropy(img2d)
    lap = laplace(img2d)
    sharpness = lap.var()
    snr = img2d.mean() / (img2d.std() + 1e-8)
    return contrast, entropy, sharpness, snr

if DATA_LOADED:
    quality_data = []
    for cls_idx in range(num_classes):
        cls_name = CLASS_LABELS.get(cls_idx, f'Class {cls_idx}')
        idx = np.where(y_train_idx == cls_idx)[0]
        
        if len(idx) > 0:
            contrasts, entropies, sharpnesses, snrs = [], [], [], []
            # Sample up to 100 images for efficiency
            for i in idx[:100]:
                c, e, s, n = compute_quality_metrics(X_train[i])
                contrasts.append(c)
                entropies.append(e)
                sharpnesses.append(s)
                snrs.append(n)
            
            quality_data.append({
                'Class': cls_name,
                'Contrast': np.mean(contrasts),
                'Entropy': np.mean(entropies),
                'Sharpness': np.mean(sharpnesses),
                'SNR': np.mean(snrs)
            })
    
    df_quality = pd.DataFrame(quality_data)
    
    # Display styled table
    print("📊 Image Quality Metrics per Dementia Stage:")
    display(df_quality.style.background_gradient(cmap='RdYlGn')
            .set_properties(**{'text-align': 'center'})
            .format({'Contrast': '{:.4f}', 'Entropy': '{:.4f}', 'Sharpness': '{:.6f}', 'SNR': '{:.4f}'}))
    
    # Radar chart for quality metrics
    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection='polar'))
    
    categories = ['Contrast', 'Entropy', 'Sharpness (log)', 'SNR']
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    
    for i, row in df_quality.iterrows():
        values = [
            row['Contrast'] / df_quality['Contrast'].max(),
            row['Entropy'] / df_quality['Entropy'].max(),
            np.log1p(row['Sharpness']) / np.log1p(df_quality['Sharpness'].max()),
            row['SNR'] / df_quality['SNR'].max()
        ]
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=row['Class'])
        ax.fill(angles, values, alpha=0.15)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=11)
    ax.set_title('🔬 Image Quality Radar Chart', fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    
    plt.tight_layout()
    plt.savefig('image_quality_radar.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ Load data first!")

## 6. Build Advanced CNN Architecture

Custom CNN with batch normalization, dropout, and architecture visualization.

In [ ]:
# ====== BUILD ADVANCED CNN ARCHITECTURE ======
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, BatchNormalization, Dropout, Flatten, Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard

# Enable mixed precision for GPU speedup
try:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print('⚡ Mixed precision training enabled (FP16)')
except Exception as e:
    print(f'ℹ️ Mixed precision not available: {e}')

def build_cnn(input_shape, num_classes):
    """
    Advanced CNN for Alzheimer's MRI Classification
    Architecture: 4 Conv Blocks + Global Average Pooling + Dense
    """
    inputs = Input(shape=input_shape)
    
    # Block 1
    x = Conv2D(32, (3,3), activation='relu', padding='same', name='conv1_1')(inputs)
    x = BatchNormalization()(x)
    x = Conv2D(32, (3,3), activation='relu', padding='same', name='conv1_2')(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.25)(x)
    
    # Block 2
    x = Conv2D(64, (3,3), activation='relu', padding='same', name='conv2_1')(x)
    x = BatchNormalization()(x)
    x = Conv2D(64, (3,3), activation='relu', padding='same', name='conv2_2')(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.25)(x)
    
    # Block 3
    x = Conv2D(128, (3,3), activation='relu', padding='same', name='conv3_1')(x)
    x = BatchNormalization()(x)
    x = Conv2D(128, (3,3), activation='relu', padding='same', name='conv3_2')(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.4)(x)
    
    # Block 4
    x = Conv2D(256, (3,3), activation='relu', padding='same', name='conv4_1')(x)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling2D()(x)
    
    # Dense layers
    x = Dense(256, activation='relu', name='dense1')(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation='relu', name='embeddings')(x)
    
    # Output layer (float32 for numerical stability with mixed precision)
    outputs = Dense(num_classes, activation='softmax', dtype='float32', name='predictions')(x)
    
    model = Model(inputs, outputs, name='NeuroXAI_CNN')
    return model

# Build model
if DATA_LOADED:
    INPUT_SHAPE = (IMG_SIZE[0], IMG_SIZE[1], 1)
    model = build_cnn(INPUT_SHAPE, num_classes)
    
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    print("\n🧠 NeuroXAI CNN Architecture Summary:")
    model.summary()
    
    # Count parameters
    trainable = np.sum([np.prod(w.shape) for w in model.trainable_weights])
    non_trainable = np.sum([np.prod(w.shape) for w in model.non_trainable_weights])
    print(f"\n📊 Model Parameters:")
    print(f"   Trainable: {trainable:,}")
    print(f"   Non-trainable: {non_trainable:,}")
    print(f"   Total: {trainable + non_trainable:,}")
else:
    print("⚠️ Load data first!")

## 7. Model Training with Callbacks

Train with EarlyStopping, checkpoints, and learning rate scheduling.

In [ ]:
# Training with callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

EPOCHS = 30
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=1)

# Save final model
model.save('alzheimer_cnn_final.keras')
print('Training complete. Best model saved.')

## 8. Training History Visualization

Loss curves, accuracy curves, and learning rate schedule.

In [ ]:
# Training History Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
hist = history.history
epochs_range = range(1, len(hist['loss'])+1)

# Loss
axes[0].plot(epochs_range, hist['loss'], 'b-', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, hist['val_loss'], 'r--', label='Val Loss', linewidth=2)
best_epoch = np.argmin(hist['val_loss']) + 1
axes[0].axvline(best_epoch, color='green', linestyle=':', label=f'Best Epoch ({best_epoch})')
axes[0].fill_between(epochs_range, hist['loss'], hist['val_loss'], alpha=0.1)
axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, hist['accuracy'], 'b-', label='Train Acc', linewidth=2)
axes[1].plot(epochs_range, hist['val_accuracy'], 'r--', label='Val Acc', linewidth=2)
axes[1].axvline(best_epoch, color='green', linestyle=':', label=f'Best Epoch')
axes[1].set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning Rate (if available)
if 'lr' in hist:
    axes[2].plot(epochs_range, hist['lr'], 'g-', linewidth=2, marker='o')
    axes[2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Learning Rate')
    axes[2].set_yscale('log')
else:
    # Simulated LR decay visualization
    lr_sim = [1e-3 * (0.5 ** (i//5)) for i in range(len(epochs_range))]
    axes[2].plot(epochs_range, lr_sim, 'g-', linewidth=2, marker='o')
    axes[2].set_title('Learning Rate Schedule (Simulated)', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Learning Rate')
    axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300)
plt.show()

## 9. Model Evaluation – Confusion Matrix

Detailed confusion matrix with normalized values and per-class accuracy.

In [ ]:
# ====== CONFUSION MATRIX ======
if DATA_LOADED and 'model' in dir():
    print("🔄 Generating predictions on test set...")
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = y_test_idx

    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    # Get class names
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=class_names_list, yticklabels=class_names_list,
                linewidths=0.5, linecolor='gray')
    axes[0].set_title('🧠 Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Predicted', fontsize=12)
    axes[0].set_ylabel('True', fontsize=12)
    axes[0].tick_params(axis='x', rotation=30)
    axes[0].tick_params(axis='y', rotation=0)

    # Normalized
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='RdYlGn', ax=axes[1],
                xticklabels=class_names_list, yticklabels=class_names_list,
                linewidths=0.5, linecolor='gray')
    axes[1].set_title('🧠 Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Predicted', fontsize=12)
    axes[1].set_ylabel('True', fontsize=12)
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].tick_params(axis='y', rotation=0)

    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Per-class accuracy
    print("\n📊 Per-Class Accuracy:")
    for i, cls_name in enumerate(class_names_list):
        acc = cm_norm[i, i] * 100
        print(f"   {cls_name}: {acc:.1f}%")
    print(f"\n   Overall Accuracy: {np.trace(cm) / cm.sum() * 100:.1f}%")
else:
    print("⚠️ Train model first!")

## 10. Classification Report Heatmap

Precision, recall, F1-score as a heatmap visualization.

In [ ]:
# ====== CLASSIFICATION REPORT HEATMAP ======
if DATA_LOADED and 'y_pred' in dir():
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    
    report = classification_report(y_true, y_pred, target_names=class_names_list, output_dict=True)
    df_report = pd.DataFrame(report).T
    df_report_classes = df_report.iloc[:num_classes]  # Only class rows
    
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(df_report_classes[['precision', 'recall', 'f1-score']], 
                annot=True, fmt='.3f', cmap='RdYlGn', ax=ax, vmin=0, vmax=1,
                linewidths=0.5, linecolor='gray')
    ax.set_title('🔬 Classification Report Heatmap', fontsize=14, fontweight='bold')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    plt.tight_layout()
    plt.savefig('classification_report_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Full report
    print("\n📊 Full Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names_list))
else:
    print("⚠️ Run predictions first!")

## 11. ROC Curves (Multi-class)

One-vs-Rest ROC curves with AUC values and micro/macro averages.

In [ ]:
# ====== ROC CURVES (MULTI-CLASS ONE-VS-REST) ======
if DATA_LOADED and 'y_pred_proba' in dir():
    from sklearn.preprocessing import label_binarize
    
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    y_test_bin = y_test  # Already one-hot encoded
    
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(num_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Micro-average
    fpr['micro'], tpr['micro'], _ = roc_curve(y_test_bin.ravel(), y_pred_proba.ravel())
    roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])

    # Macro-average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(num_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(num_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= num_classes
    fpr['macro'], tpr['macro'] = all_fpr, mean_tpr
    roc_auc['macro'] = auc(fpr['macro'], tpr['macro'])

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.viridis(np.linspace(0, 1, num_classes))
    
    for i, (cls_name, color) in enumerate(zip(class_names_list, colors)):
        ax.plot(fpr[i], tpr[i], color=color, lw=2, label=f'{cls_name} (AUC = {roc_auc[i]:.3f})')
    
    ax.plot(fpr['micro'], tpr['micro'], color='deeppink', linestyle=':', lw=3, 
            label=f'Micro-avg (AUC = {roc_auc["micro"]:.3f})')
    ax.plot(fpr['macro'], tpr['macro'], color='navy', linestyle='--', lw=3, 
            label=f'Macro-avg (AUC = {roc_auc["macro"]:.3f})')
    ax.plot([0,1], [0,1], 'k--', lw=1, alpha=0.5)
    
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('🧠 Multi-class ROC Curves (One-vs-Rest)', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('roc_curves.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n📊 AUC Scores:")
    for i, cls_name in enumerate(class_names_list):
        print(f"   {cls_name}: {roc_auc[i]:.4f}")
    print(f"   Macro-average: {roc_auc['macro']:.4f}")
    print(f"   Micro-average: {roc_auc['micro']:.4f}")
else:
    print("⚠️ Run predictions first!")

## 12. Precision-Recall Curves

PR curves with average precision scores and iso-F1 curves.

In [ ]:
# ====== PRECISION-RECALL CURVES ======
if DATA_LOADED and 'y_pred_proba' in dir():
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    
    precision_dict, recall_dict, ap = {}, {}, {}
    for i in range(num_classes):
        precision_dict[i], recall_dict[i], _ = precision_recall_curve(y_test_bin[:, i], y_pred_proba[:, i])
        ap[i] = average_precision_score(y_test_bin[:, i], y_pred_proba[:, i])

    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Iso-F1 curves
    f_scores = np.linspace(0.2, 0.8, num=4)
    for f in f_scores:
        x = np.linspace(0.01, 1)
        y_f = f * x / (2 * x - f)
        ax.plot(x[y_f >= 0], y_f[y_f >= 0], color='gray', alpha=0.2)
        ax.annotate(f'F1={f:.1f}', xy=(0.9, y_f[45] + 0.02), fontsize=8, color='gray')

    colors = plt.cm.plasma(np.linspace(0, 1, num_classes))
    for i, (cls_name, color) in enumerate(zip(class_names_list, colors)):
        ax.plot(recall_dict[i], precision_dict[i], color=color, lw=2, label=f'{cls_name} (AP = {ap[i]:.3f})')

    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title('🔬 Precision-Recall Curves with Iso-F1', fontsize=14, fontweight='bold')
    ax.legend(loc='lower left')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('precision_recall_curves.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n📊 Average Precision Scores:")
    for i, cls_name in enumerate(class_names_list):
        print(f"   {cls_name}: {ap[i]:.4f}")
else:
    print("⚠️ Run predictions first!")

## 13. AUC Comparison Bar Chart

Compare AUC-ROC and AUC-PR scores across all classes.

In [ ]:
# ====== AUC COMPARISON BAR CHART ======
if DATA_LOADED and 'roc_auc' in dir() and 'ap' in dir():
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    
    auc_roc_vals = [roc_auc[i] for i in range(num_classes)]
    auc_pr_vals = [ap[i] for i in range(num_classes)]

    x = np.arange(num_classes)
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 6))
    bars1 = ax.bar(x - width/2, auc_roc_vals, width, label='AUC-ROC', color='steelblue', edgecolor='black')
    bars2 = ax.bar(x + width/2, auc_pr_vals, width, label='AUC-PR', color='coral', edgecolor='black')

    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('🔬 AUC-ROC vs AUC-PR Comparison by Class', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names_list, rotation=15)
    ax.legend()
    ax.set_ylim([0, 1.1])
    ax.grid(axis='y', alpha=0.3)

    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)
    for bar in bars2:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

    plt.tight_layout()
    plt.savefig('auc_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ Run ROC and PR curves first!")

## 14. SHAP GradientExplainer Setup

Initialize SHAP with diverse background samples for explainability.

In [ ]:
# SHAP GradientExplainer Setup
import shap
shap.initjs()

# Create diverse background dataset (100 samples balanced across classes)
background_idx = []
samples_per_class = 100 // num_classes
for i in range(num_classes):
    cls_idx = np.where(np.argmax(y_train, axis=1) == i)[0]
    background_idx.extend(np.random.choice(cls_idx, min(samples_per_class, len(cls_idx)), replace=False))
background = X_train[background_idx]
print(f'Background dataset: {background.shape}')

# Initialize SHAP GradientExplainer
explainer = shap.GradientExplainer(model, background)
print('SHAP GradientExplainer initialized')

# Compute SHAP values for test subset
test_subset_idx = np.random.choice(len(X_test), min(50, len(X_test)), replace=False)
test_subset = X_test[test_subset_idx]
shap_values = explainer.shap_values(test_subset)
print(f'SHAP values computed: {len(shap_values)} classes, shape per class: {shap_values[0].shape}')

## 15. SHAP Summary Plot – Global Feature Importance

Global feature importance across all pixel locations for each class.

In [ ]:
# ====== SHAP SUMMARY PLOT - GLOBAL FEATURE IMPORTANCE ======
if 'shap_values' in dir():
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    
    # Reshape SHAP values for visualization
    shap_flat = [sv.reshape(sv.shape[0], -1) for sv in shap_values]
    test_flat = test_subset.reshape(test_subset.shape[0], -1)

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    for i in range(min(4, num_classes)):
        cls_name = class_names_list[i]
        ax = axes[i//2, i%2]
        plt.sca(ax)
        shap.summary_plot(shap_flat[i], test_flat, show=False, max_display=20)
        ax.set_title(f'🧠 SHAP Summary: {cls_name}', fontsize=12, fontweight='bold')
    
    plt.suptitle('Global SHAP Feature Importance by Dementia Stage', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('shap_summary_global.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ Run SHAP explainer first!")

## 16. SHAP Image Heatmaps – Per-Sample Explainability

Overlay SHAP values on MRI scans for individual predictions.

In [ ]:
# ====== SHAP IMAGE HEATMAPS - PER-SAMPLE EXPLAINABILITY ======
def plot_shap_heatmap(img, shap_val, pred_class, true_class, ax):
    """Plot SHAP heatmap overlaid on MRI image"""
    shap_2d = shap_val.squeeze()
    img_2d = img.squeeze()
    ax.imshow(img_2d, cmap='gray', alpha=0.7)
    im = ax.imshow(shap_2d, cmap='RdBu_r', alpha=0.5, vmin=-np.abs(shap_2d).max(), vmax=np.abs(shap_2d).max())
    color = 'green' if pred_class == true_class else 'red'
    ax.set_title(f'Pred: {pred_class}\nTrue: {true_class}', fontsize=10, color=color)
    ax.axis('off')
    return im

if 'shap_values' in dir() and 'model' in dir():
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    
    # Sample predictions
    pred_subset = model.predict(test_subset, verbose=0)
    pred_labels_idx = np.argmax(pred_subset, axis=1)
    true_labels_idx = np.argmax(y_test[test_subset_idx], axis=1)
    
    pred_labels = [class_names_list[i] if i < len(class_names_list) else f'Class {i}' for i in pred_labels_idx]
    true_labels = [class_names_list[i] if i < len(class_names_list) else f'Class {i}' for i in true_labels_idx]

    # Plot 4x4 grid
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    for i in range(min(16, len(test_subset))):
        ax = axes[i//4, i%4]
        pred_cls_idx = pred_labels_idx[i]
        im = plot_shap_heatmap(test_subset[i], shap_values[pred_cls_idx][i], pred_labels[i], true_labels[i], ax)
    
    fig.colorbar(im, ax=axes, orientation='vertical', fraction=0.02, pad=0.04, label='SHAP Value')
    fig.suptitle('🧠 SHAP Heatmaps: Feature Importance per Pixel', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('shap_heatmaps.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ SHAP heatmaps generated!")
else:
    print("⚠️ Run SHAP explainer first!")

## 17. SHAP Force Plot – Single Prediction

Interactive force plots showing positive and negative feature contributions.

In [ ]:
# SHAP Force Plot - Single Prediction
sample_idx = 0
pred_cls_idx = np.argmax(pred_subset[sample_idx])
base_value = explainer.expected_value[pred_cls_idx] if hasattr(explainer.expected_value, '__len__') else explainer.expected_value

# Force plot (static)
shap.force_plot(base_value, shap_flat[pred_cls_idx][sample_idx], test_flat[sample_idx], matplotlib=True, show=False)
plt.title(f'SHAP Force Plot: Sample {sample_idx} → {pred_labels[sample_idx]}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_force_plot.png', dpi=300, bbox_inches='tight')
plt.show()

## 18. SHAP Bar Plot – Mean Absolute SHAP Values

Top contributing brain regions by mean |SHAP| values.

In [ ]:
# ====== SHAP BAR PLOT - MEAN ABSOLUTE SHAP VALUES PER CLASS ======
if 'shap_flat' in dir():
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    for i in range(min(4, num_classes)):
        ax = axes[i//2, i%2]
        mean_abs_shap = np.abs(shap_flat[i]).mean(axis=0)
        top_idx = np.argsort(mean_abs_shap)[-20:][::-1]
        ax.barh(range(20), mean_abs_shap[top_idx], color=plt.cm.viridis(np.linspace(0.2, 0.8, 20)))
        ax.set_yticks(range(20))
        ax.set_yticklabels([f'Pixel {idx}' for idx in top_idx])
        ax.set_xlabel('Mean |SHAP|')
        ax.set_title(f'🧠 Top Features: {class_names_list[i]}', fontsize=12, fontweight='bold')
        ax.invert_yaxis()
    
    plt.suptitle('SHAP Feature Importance by Dementia Stage', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('shap_bar_plot.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ Run SHAP analysis first!")

## 19. Grad-CAM Visualization

Gradient-weighted Class Activation Mapping for CNN focus regions.

In [ ]:
# ====== GRAD-CAM IMPLEMENTATION ======
def get_grad_cam(model, img, class_idx, layer_name=None):
    """Generate Grad-CAM heatmap for given image and class"""
    # Find last conv layer if not specified
    if layer_name is None:
        for layer in reversed(model.layers):
            if 'conv' in layer.name.lower():
                layer_name = layer.name
                break
    
    grad_model = Model([model.inputs], [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(np.expand_dims(img, 0))
        loss = predictions[:, class_idx]
    
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    heatmap = cv2.resize(heatmap.numpy(), IMG_SIZE)
    return heatmap

def overlay_gradcam(img, heatmap, alpha=0.4):
    """Overlay Grad-CAM heatmap on original image"""
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB) / 255.0
    img_rgb = np.stack([img.squeeze()]*3, axis=-1)
    overlay = alpha * heatmap_colored + (1 - alpha) * img_rgb
    return np.clip(overlay, 0, 1)

# Generate Grad-CAM for test samples
if DATA_LOADED and 'model' in dir():
    print("🔄 Generating Grad-CAM visualizations...")
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    for i in range(16):
        ax = axes[i//4, i%4]
        img = test_subset[i]
        pred_cls = np.argmax(pred_subset[i])
        true_cls = true_labels_idx[i]
        
        heatmap = get_grad_cam(model, img, pred_cls)
        overlay = overlay_gradcam(img, heatmap)
        
        ax.imshow(overlay)
        pred_name = class_names_list[pred_cls] if pred_cls < len(class_names_list) else f'Class {pred_cls}'
        true_name = class_names_list[true_cls] if true_cls < len(class_names_list) else f'Class {true_cls}'
        
        color = 'green' if pred_cls == true_cls else 'red'
        ax.set_title(f'Pred: {pred_name}\nTrue: {true_name}', fontsize=9, color=color)
        ax.axis('off')
    
    fig.suptitle('🧠 Grad-CAM: CNN Focus Regions for Alzheimer\'s Detection', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('gradcam_heatmaps.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Grad-CAM visualization complete!")
else:
    print("⚠️ Load data and train model first!")

## 20. Integrated Gradients Visualization

Attribution by accumulating gradients along path from baseline.

In [ ]:
# Integrated Gradients Implementation
@tf.function
def compute_gradients(model, img, target_class):
    with tf.GradientTape() as tape:
        tape.watch(img)
        preds = model(img)
        loss = preds[:, target_class]
    return tape.gradient(loss, img)

def integrated_gradients(model, img, target_class, baseline=None, steps=50):
    if baseline is None:
        baseline = np.zeros_like(img)
    
    # Generate interpolated images
    alphas = np.linspace(0, 1, steps + 1)
    interpolated = np.array([baseline + alpha * (img - baseline) for alpha in alphas])
    interpolated = interpolated.astype(np.float32)
    
    # Compute gradients
    grads = []
    for interp in interpolated:
        g = compute_gradients(model, tf.constant([interp]), target_class)
        grads.append(g.numpy()[0])
    grads = np.array(grads)
    
    # Approximate integral using trapezoidal rule
    avg_grads = np.mean(grads, axis=0)
    ig = (img - baseline) * avg_grads
    return ig

# Compute IG for samples
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for i in range(16):
    ax = axes[i//4, i%4]
    img = test_subset[i]
    pred_cls = np.argmax(pred_subset[i])
    ig = integrated_gradients(model, img, pred_cls, steps=30)
    ig_2d = ig.squeeze()
    ax.imshow(img.squeeze(), cmap='gray', alpha=0.7)
    ax.imshow(ig_2d, cmap='RdBu_r', alpha=0.5, vmin=-np.abs(ig_2d).max(), vmax=np.abs(ig_2d).max())
    ax.set_title(f'Pred: {pred_labels[i]}', fontsize=10)
    ax.axis('off')
fig.suptitle('Integrated Gradients Attribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('integrated_gradients.png', dpi=300, bbox_inches='tight')
plt.show()

## 21. Occlusion Sensitivity Maps

Systematically occlude regions to measure prediction changes.

In [ ]:
# Occlusion Sensitivity Maps
def occlusion_sensitivity(model, img, target_class, patch_size=16, stride=8):
    h, w = img.shape[:2]
    sensitivity_map = np.zeros((h, w))
    count_map = np.zeros((h, w))
    base_pred = model.predict(np.expand_dims(img, 0), verbose=0)[0, target_class]
    
    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            occluded = img.copy()
            occluded[y:y+patch_size, x:x+patch_size] = 0  # Black patch
            pred = model.predict(np.expand_dims(occluded, 0), verbose=0)[0, target_class]
            diff = base_pred - pred  # Positive = important region
            sensitivity_map[y:y+patch_size, x:x+patch_size] += diff
            count_map[y:y+patch_size, x:x+patch_size] += 1
    
    sensitivity_map /= np.maximum(count_map, 1)
    return sensitivity_map

# Generate occlusion maps for 4 samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img = test_subset[i]
    pred_cls = np.argmax(pred_subset[i])
    occ_map = occlusion_sensitivity(model, img, pred_cls, patch_size=12, stride=6)
    
    # Original
    axes[0, i].imshow(img.squeeze(), cmap='gray')
    axes[0, i].set_title(f'Original: {pred_labels[i]}', fontsize=10)
    axes[0, i].axis('off')
    
    # Occlusion map
    axes[1, i].imshow(img.squeeze(), cmap='gray', alpha=0.6)
    im = axes[1, i].imshow(occ_map, cmap='hot', alpha=0.5)
    axes[1, i].set_title('Occlusion Sensitivity', fontsize=10)
    axes[1, i].axis('off')

fig.colorbar(im, ax=axes, orientation='vertical', fraction=0.02, pad=0.04, label='Importance')
fig.suptitle('Occlusion Sensitivity Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('occlusion_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

## 22. Layer-wise Activation Visualization

Visualize activations at each convolutional layer.

In [ ]:
# Layer-wise Activation Visualization
conv_layers = [layer for layer in model.layers if 'conv' in layer.name.lower()]
print(f'Found {len(conv_layers)} conv layers:', [l.name for l in conv_layers])

# Create activation model
activation_model = Model(inputs=model.input, outputs=[l.output for l in conv_layers])

# Get activations for one sample
sample_img = test_subset[0:1]
activations = activation_model.predict(sample_img, verbose=0)

# Plot activations from each layer (first 8 filters)
fig, axes = plt.subplots(len(conv_layers), 8, figsize=(16, len(conv_layers)*2))
for i, (activation, layer) in enumerate(zip(activations, conv_layers)):
    for j in range(min(8, activation.shape[-1])):
        axes[i, j].imshow(activation[0, :, :, j], cmap='viridis')
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_ylabel(layer.name, fontsize=8, rotation=90, labelpad=10)
fig.suptitle('Layer-wise Activations (First 8 Filters per Layer)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('layer_activations.png', dpi=300, bbox_inches='tight')
plt.show()

## 23. Convolutional Filter Visualization

Visualize learned convolutional filters as image grids.

In [ ]:
# Convolutional Filter Visualization
first_conv = conv_layers[0]
filters, biases = first_conv.get_weights()
print(f'First conv layer filters shape: {filters.shape}')

# Normalize filters for visualization
f_min, f_max = filters.min(), filters.max()
filters_norm = (filters - f_min) / (f_max - f_min + 1e-8)

# Plot filters
n_filters = filters.shape[-1]
n_cols = 8
n_rows = (n_filters + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*2))
for i in range(n_filters):
    ax = axes[i//n_cols, i%n_cols]
    ax.imshow(filters_norm[:, :, 0, i], cmap='gray')
    ax.axis('off')
    ax.set_title(f'F{i}', fontsize=8)
# Hide unused axes
for i in range(n_filters, n_rows*n_cols):
    axes[i//n_cols, i%n_cols].axis('off')
fig.suptitle('First Conv Layer Filters (Learned Patterns)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('conv_filters.png', dpi=300, bbox_inches='tight')
plt.show()

## 24. t-SNE Embedding of CNN Features

2D/3D t-SNE visualization of extracted features colored by class.

In [ ]:
# t-SNE Embedding of CNN Features
from sklearn.manifold import TSNE

# Extract embeddings from penultimate layer
embedding_model = Model(inputs=model.input, outputs=model.get_layer('embeddings').output)
embeddings_train = embedding_model.predict(X_train, verbose=0)
embeddings_test = embedding_model.predict(X_test, verbose=0)
print(f'Embeddings shape: {embeddings_train.shape}')

# t-SNE 2D
tsne_2d = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
emb_2d = tsne_2d.fit_transform(embeddings_test)

# t-SNE 3D
tsne_3d = TSNE(n_components=3, perplexity=30, random_state=SEED, n_iter=1000)
emb_3d = tsne_3d.fit_transform(embeddings_test)

y_test_labels = le.inverse_transform(np.argmax(y_test, axis=1))

# 2D Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for i, cls in enumerate(le.classes_):
    idx = np.where(y_test_labels == cls)[0]
    axes[0].scatter(emb_2d[idx, 0], emb_2d[idx, 1], label=cls, alpha=0.7, s=50)
axes[0].set_title('t-SNE 2D Embedding', fontsize=14, fontweight='bold')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 3D Plot (as 2D projection)
ax3d = fig.add_subplot(1, 2, 2, projection='3d')
for i, cls in enumerate(le.classes_):
    idx = np.where(y_test_labels == cls)[0]
    ax3d.scatter(emb_3d[idx, 0], emb_3d[idx, 1], emb_3d[idx, 2], label=cls, alpha=0.7, s=50)
ax3d.set_title('t-SNE 3D Embedding', fontsize=14, fontweight='bold')
ax3d.set_xlabel('t-SNE 1')
ax3d.set_ylabel('t-SNE 2')
ax3d.set_zlabel('t-SNE 3')
ax3d.legend()
plt.tight_layout()
plt.savefig('tsne_embeddings.png', dpi=300)
plt.show()

## 25. UMAP Embedding Visualization

UMAP dimensionality reduction with cluster separation analysis.

In [ ]:
# ====== UMAP EMBEDDING VISUALIZATION ======
import umap

if DATA_LOADED and 'embeddings_test' in dir():
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    
    print("🔄 Computing UMAP embeddings...")
    
    # UMAP 2D
    umap_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED)
    umap_emb_2d = umap_2d.fit_transform(embeddings_test)

    # UMAP 3D
    umap_3d = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=SEED)
    umap_emb_3d = umap_3d.fit_transform(embeddings_test)

    fig = plt.figure(figsize=(16, 6))
    
    # 2D Plot
    ax1 = fig.add_subplot(1, 2, 1)
    for i in range(num_classes):
        cls_name = class_names_list[i] if i < len(class_names_list) else f'Class {i}'
        idx = np.where(y_test_idx == i)[0]
        ax1.scatter(umap_emb_2d[idx, 0], umap_emb_2d[idx, 1], label=cls_name, alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
    ax1.set_title('🧠 UMAP 2D Embedding', fontsize=14, fontweight='bold')
    ax1.set_xlabel('UMAP 1')
    ax1.set_ylabel('UMAP 2')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # 3D Plot
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    for i in range(num_classes):
        cls_name = class_names_list[i] if i < len(class_names_list) else f'Class {i}'
        idx = np.where(y_test_idx == i)[0]
        ax2.scatter(umap_emb_3d[idx, 0], umap_emb_3d[idx, 1], umap_emb_3d[idx, 2], label=cls_name, alpha=0.7, s=50)
    ax2.set_title('🧠 UMAP 3D Embedding', fontsize=14, fontweight='bold')
    ax2.set_xlabel('UMAP 1')
    ax2.set_ylabel('UMAP 2')
    ax2.set_zlabel('UMAP 3')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig('umap_embeddings.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ UMAP embeddings computed!")
else:
    print("⚠️ Extract embeddings first!")

## 26. PCA Analysis of Extracted Features

Principal Component Analysis with explained variance and visualization.

In [ ]:
# ====== PCA ANALYSIS OF EXTRACTED FEATURES ======
from sklearn.decomposition import PCA

if DATA_LOADED and 'embeddings_test' in dir():
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    
    pca = PCA(n_components=min(50, embeddings_test.shape[1]))
    pca_emb = pca.fit_transform(embeddings_test)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Explained variance
    axes[0].bar(range(1, len(pca.explained_variance_ratio_)+1), pca.explained_variance_ratio_, 
                color='steelblue', alpha=0.7, edgecolor='black')
    axes[0].plot(range(1, len(pca.explained_variance_ratio_)+1), np.cumsum(pca.explained_variance_ratio_), 
                 'r-o', label='Cumulative', linewidth=2)
    axes[0].axhline(0.95, color='green', linestyle='--', label='95% threshold')
    axes[0].set_xlabel('Principal Component')
    axes[0].set_ylabel('Explained Variance Ratio')
    axes[0].set_title('🔬 PCA Explained Variance', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # PCA 2D scatter
    for i in range(num_classes):
        cls_name = class_names_list[i] if i < len(class_names_list) else f'Class {i}'
        idx = np.where(y_test_idx == i)[0]
        axes[1].scatter(pca_emb[idx, 0], pca_emb[idx, 1], label=cls_name, alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
    axes[1].set_xlabel('PC 1')
    axes[1].set_ylabel('PC 2')
    axes[1].set_title('🔬 PCA 2D Projection', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # PCA component heatmap
    sns.heatmap(pca.components_[:10, :20], cmap='RdBu_r', ax=axes[2], 
                xticklabels=False, yticklabels=[f'PC{i+1}' for i in range(10)])
    axes[2].set_title('🔬 Top 10 Principal Components', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Feature Index')

    plt.tight_layout()
    plt.savefig('pca_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print variance explained
    print(f"\n📊 PCA Summary:")
    print(f"   Components for 95% variance: {np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.95) + 1}")
    print(f"   Total explained variance: {pca.explained_variance_ratio_.sum():.2%}")
else:
    print("⚠️ Extract embeddings first!")

## 27. Hierarchical Clustering Dendrogram

Cluster analysis of embeddings with dendrogram visualization.

In [ ]:
# Hierarchical Clustering Dendrogram
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

# Use subset for clustering (100 samples)
subset_idx = np.random.choice(len(embeddings_test), min(100, len(embeddings_test)), replace=False)
emb_subset = embeddings_test[subset_idx]
labels_subset = y_test_labels[subset_idx]

# Compute linkage
linkage_matrix = linkage(emb_subset, method='ward')

# Plot dendrogram
fig, ax = plt.subplots(figsize=(16, 8))
color_map = {cls: plt.cm.viridis(i/len(le.classes_)) for i, cls in enumerate(le.classes_)}
dendrogram(linkage_matrix, labels=labels_subset, leaf_rotation=90, leaf_font_size=8, ax=ax)
ax.set_title('Hierarchical Clustering Dendrogram', fontsize=14, fontweight='bold')
ax.set_xlabel('Sample (Class)')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.savefig('dendrogram.png', dpi=300, bbox_inches='tight')
plt.show()

## 28. Radar Chart – Model Performance Metrics

Spider chart comparing accuracy, precision, recall, F1, and specificity.

In [ ]:
# Radar Chart - Model Performance Metrics
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def compute_specificity(y_true, y_pred, num_classes):
    specificities = []
    for i in range(num_classes):
        tn = np.sum((y_true != i) & (y_pred != i))
        fp = np.sum((y_true != i) & (y_pred == i))
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        specificities.append(spec)
    return specificities

# Compute metrics per class
metrics_per_class = []
specificities = compute_specificity(y_true, y_pred, num_classes)
for i, cls in enumerate(le.classes_):
    y_true_bin = (y_true == i).astype(int)
    y_pred_bin = (y_pred == i).astype(int)
    metrics_per_class.append({
        'Class': cls,
        'Accuracy': accuracy_score(y_true_bin, y_pred_bin),
        'Precision': precision_score(y_true_bin, y_pred_bin, zero_division=0),
        'Recall': recall_score(y_true_bin, y_pred_bin, zero_division=0),
        'F1-Score': f1_score(y_true_bin, y_pred_bin, zero_division=0),
        'Specificity': specificities[i]
    })

# Radar chart
categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']
n_cats = len(categories)
angles = [n / float(n_cats) * 2 * np.pi for n in range(n_cats)]
angles += angles[:1]  # Complete the loop

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
colors = plt.cm.Set2(np.linspace(0, 1, num_classes))

for i, metrics in enumerate(metrics_per_class):
    values = [metrics[cat] for cat in categories]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=metrics['Class'], color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Model Performance Radar Chart', fontsize=14, fontweight='bold', y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
plt.tight_layout()
plt.savefig('radar_chart.png', dpi=300, bbox_inches='tight')
plt.show()

## 29. Violin Plot – Prediction Confidence by Class

Distribution of prediction confidence scores for each dementia stage.

In [ ]:
# Violin Plot - Prediction Confidence by Class
max_probs = np.max(y_pred_proba, axis=1)
df_conf = pd.DataFrame({
    'True Class': y_test_labels,
    'Predicted Class': pred_labels,
    'Confidence': max_probs,
    'Correct': (y_test_labels == pred_labels)
})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# By True Class
sns.violinplot(data=df_conf, x='True Class', y='Confidence', ax=axes[0], palette='viridis', inner='box')
axes[0].set_title('Prediction Confidence by True Class', fontsize=14, fontweight='bold')
axes[0].set_xlabel('True Dementia Stage')
axes[0].set_ylabel('Confidence')

# By Correctness
sns.violinplot(data=df_conf, x='True Class', y='Confidence', hue='Correct', split=True, ax=axes[1], palette=['salmon', 'lightgreen'], inner='quart')
axes[1].set_title('Confidence: Correct vs Incorrect', fontsize=14, fontweight='bold')
axes[1].set_xlabel('True Dementia Stage')
axes[1].set_ylabel('Confidence')
axes[1].legend(title='Correct')

plt.tight_layout()
plt.savefig('violin_confidence.png', dpi=300)
plt.show()

## 30. Interactive 3D Scatter Plot (Plotly)

Interactive 3D visualization of feature space with hover information.

In [ ]:
# Interactive 3D Scatter Plot (Plotly)
import plotly.express as px

df_3d = pd.DataFrame({
    'UMAP1': umap_emb_3d[:, 0],
    'UMAP2': umap_emb_3d[:, 1],
    'UMAP3': umap_emb_3d[:, 2],
    'True Class': y_test_labels,
    'Predicted Class': pred_labels,
    'Confidence': max_probs,
    'Correct': ['Correct' if c else 'Incorrect' for c in (y_test_labels == pred_labels)]
})

fig = px.scatter_3d(df_3d, x='UMAP1', y='UMAP2', z='UMAP3', color='True Class',
                    symbol='Correct', size='Confidence', size_max=15,
                    hover_data=['Predicted Class', 'Confidence'],
                    title='Interactive 3D UMAP Embedding',
                    color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01))
fig.write_html('interactive_3d_umap.html')
fig.show()

## 31. Sankey Diagram – Prediction Flow

Flow from true labels to predicted labels showing misclassification patterns.

In [ ]:
# ====== T-SNE EMBEDDING VISUALIZATION ======
from sklearn.manifold import TSNE

if DATA_LOADED and 'embeddings_test' in dir():
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    y_test_labels = [class_names_list[i] if i < len(class_names_list) else f'Class {i}' for i in y_test_idx]
    
    print("🔄 Computing t-SNE embeddings (this may take a moment)...")
    
    # t-SNE 2D
    tsne_2d = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=SEED)
    emb_2d = tsne_2d.fit_transform(embeddings_test)
    
    # t-SNE 3D
    tsne_3d = TSNE(n_components=3, perplexity=30, n_iter=1000, random_state=SEED)
    emb_3d = tsne_3d.fit_transform(embeddings_test)

    fig = plt.figure(figsize=(16, 6))
    
    # 2D Plot
    ax1 = fig.add_subplot(1, 2, 1)
    for i in range(num_classes):
        cls_name = class_names_list[i] if i < len(class_names_list) else f'Class {i}'
        idx = np.where(y_test_idx == i)[0]
        ax1.scatter(emb_2d[idx, 0], emb_2d[idx, 1], label=cls_name, alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
    ax1.set_title('🧠 t-SNE 2D Embedding', fontsize=14, fontweight='bold')
    ax1.set_xlabel('t-SNE 1')
    ax1.set_ylabel('t-SNE 2')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # 3D Plot
    ax3d = fig.add_subplot(1, 2, 2, projection='3d')
    for i in range(num_classes):
        cls_name = class_names_list[i] if i < len(class_names_list) else f'Class {i}'
        idx = np.where(y_test_idx == i)[0]
        ax3d.scatter(emb_3d[idx, 0], emb_3d[idx, 1], emb_3d[idx, 2], label=cls_name, alpha=0.7, s=50)
    ax3d.set_title('🧠 t-SNE 3D Embedding', fontsize=14, fontweight='bold')
    ax3d.set_xlabel('t-SNE 1')
    ax3d.set_ylabel('t-SNE 2')
    ax3d.set_zlabel('t-SNE 3')
    ax3d.legend()
    
    plt.tight_layout()
    plt.savefig('tsne_embeddings.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ t-SNE embeddings computed!")
else:
    print("⚠️ Extract embeddings first!")

## 32. Sunburst Chart – Hierarchical Prediction Analysis

Hierarchical view of true class → prediction → correctness.

In [ ]:
# Sunburst Chart - Hierarchical Prediction Analysis
df_sunburst = pd.DataFrame({
    'True': y_test_labels,
    'Predicted': pred_labels,
    'Result': ['Correct' if t == p else 'Incorrect' for t, p in zip(y_test_labels, pred_labels)]
})
df_sunburst['Count'] = 1
df_agg = df_sunburst.groupby(['True', 'Predicted', 'Result']).count().reset_index()

fig = px.sunburst(df_agg, path=['True', 'Predicted', 'Result'], values='Count',
                  color='Result', color_discrete_map={'Correct': '#4CAF50', 'Incorrect': '#F44336'},
                  title='Sunburst: True → Predicted → Result')
fig.write_html('sunburst_predictions.html')
fig.show()

## 33. Parallel Coordinates Plot

Multi-metric view with class-based coloring.

In [ ]:
# Parallel Coordinates Plot
df_parallel = pd.DataFrame({
    'PC1': pca_emb[:, 0],
    'PC2': pca_emb[:, 1],
    'PC3': pca_emb[:, 2],
    'Confidence': max_probs,
    'Class': le.transform(y_test_labels)
})

fig = px.parallel_coordinates(df_parallel, color='Class',
                              dimensions=['PC1', 'PC2', 'PC3', 'Confidence'],
                              color_continuous_scale=px.colors.diverging.Tealrose,
                              title='Parallel Coordinates: Feature Space')
fig.write_html('parallel_coordinates.html')
fig.show()

## 34. Clinical Dashboard – Side-by-Side Comparison

Comprehensive panel: MRI, Grad-CAM, SHAP, and prediction probabilities.

In [ ]:
# ====== CLINICAL DASHBOARD - SIDE-BY-SIDE COMPARISON ======
def clinical_dashboard(img, model, explainer, shap_values_single, class_names, idx=0):
    """Generate comprehensive clinical dashboard for a single sample"""
    fig = plt.figure(figsize=(20, 5))
    gs = fig.add_gridspec(1, 5, width_ratios=[1, 1, 1, 1, 1.2])
    
    img_2d = img.squeeze()
    pred = model.predict(np.expand_dims(img, 0), verbose=0)[0]
    pred_cls = np.argmax(pred)
    
    # Original MRI
    ax0 = fig.add_subplot(gs[0])
    ax0.imshow(img_2d, cmap='gray')
    ax0.set_title('Original MRI', fontsize=12, fontweight='bold')
    ax0.axis('off')
    
    # Grad-CAM
    ax1 = fig.add_subplot(gs[1])
    heatmap = get_grad_cam(model, img, pred_cls)
    overlay = overlay_gradcam(img, heatmap)
    ax1.imshow(overlay)
    ax1.set_title('Grad-CAM', fontsize=12, fontweight='bold')
    ax1.axis('off')
    
    # SHAP
    ax2 = fig.add_subplot(gs[2])
    shap_2d = shap_values_single.squeeze()
    ax2.imshow(img_2d, cmap='gray', alpha=0.7)
    ax2.imshow(shap_2d, cmap='RdBu_r', alpha=0.5, vmin=-np.abs(shap_2d).max(), vmax=np.abs(shap_2d).max())
    ax2.set_title('SHAP Heatmap', fontsize=12, fontweight='bold')
    ax2.axis('off')
    
    # Integrated Gradients
    ax3 = fig.add_subplot(gs[3])
    ig = integrated_gradients(model, img, pred_cls, steps=20)
    ig_2d = ig.squeeze()
    ax3.imshow(img_2d, cmap='gray', alpha=0.7)
    ax3.imshow(ig_2d, cmap='hot', alpha=0.5)
    ax3.set_title('Integrated Gradients', fontsize=12, fontweight='bold')
    ax3.axis('off')
    
    # Prediction Probabilities
    ax4 = fig.add_subplot(gs[4])
    colors = ['#4CAF50' if i == pred_cls else '#2196F3' for i in range(len(class_names))]
    bars = ax4.barh(class_names, pred, color=colors, edgecolor='black')
    ax4.set_xlim(0, 1)
    ax4.set_xlabel('Probability', fontsize=10)
    ax4.set_title(f'Prediction: {class_names[pred_cls]}', fontsize=12, fontweight='bold')
    for bar, p in zip(bars, pred):
        ax4.text(p + 0.02, bar.get_y() + bar.get_height()/2, f'{p:.2%}', va='center', fontsize=9)
    
    plt.suptitle(f'🏥 NeuroXAI Clinical Dashboard - Sample {idx}', fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    return fig

# Generate dashboards
if DATA_LOADED and 'model' in dir() and 'shap_values' in dir():
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    
    print("🔄 Generating clinical dashboards...")
    for i in range(min(4, len(test_subset))):
        pred_cls_i = np.argmax(pred_subset[i])
        fig = clinical_dashboard(test_subset[i], model, explainer, shap_values[pred_cls_i][i], class_names_list, i)
        fig.savefig(f'clinical_dashboard_{i}.png', dpi=300, bbox_inches='tight')
        plt.show()
    
    print("✅ Clinical dashboards generated!")
else:
    print("⚠️ Run full pipeline first!")

## 35. Global Pattern Analysis per Class

Mean SHAP heatmaps showing which brain regions drive each class prediction.

In [ ]:
# ====== GLOBAL PATTERN ANALYSIS PER CLASS ======
if DATA_LOADED and 'shap_values' in dir():
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    
    fig, axes = plt.subplots(2, num_classes, figsize=(5*num_classes, 10))
    
    for i in range(num_classes):
        cls_name = class_names_list[i] if i < len(class_names_list) else f'Class {i}'
        
        # Mean SHAP for this class
        mean_shap = np.mean(np.abs(shap_values[i]), axis=0).squeeze()
        mean_img = np.mean(test_subset, axis=0).squeeze()
        
        # Original mean image
        ax0 = axes[0, i]
        ax0.imshow(mean_img, cmap='gray')
        ax0.set_title(f'Mean Image\n{cls_name}', fontsize=11, fontweight='bold')
        ax0.axis('off')
        
        # Mean SHAP overlay
        ax1 = axes[1, i]
        ax1.imshow(mean_img, cmap='gray', alpha=0.6)
        im = ax1.imshow(mean_shap, cmap='hot', alpha=0.6)
        ax1.set_title(f'Mean |SHAP|\n{cls_name}', fontsize=11, fontweight='bold')
        ax1.axis('off')

    fig.colorbar(im, ax=axes, orientation='vertical', fraction=0.02, pad=0.04, label='Mean |SHAP|')
    fig.suptitle('🧠 Global Pattern Analysis: What Drives Each Dementia Stage?', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('global_pattern_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ Run SHAP analysis first!")

## 36. Calibration Curve (Reliability Diagram)

Assess how well predicted probabilities match actual outcomes.

In [ ]:
# ====== CALIBRATION CURVE (RELIABILITY DIAGRAM) ======
from sklearn.calibration import calibration_curve

if DATA_LOADED and 'y_pred_proba' in dir():
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Per-class calibration
    for i in range(num_classes):
        cls_name = class_names_list[i] if i < len(class_names_list) else f'Class {i}'
        y_true_bin = (y_true == i).astype(int)
        y_prob = y_pred_proba[:, i]
        prob_true, prob_pred = calibration_curve(y_true_bin, y_prob, n_bins=10, strategy='uniform')
        axes[0].plot(prob_pred, prob_true, 's-', label=cls_name)

    axes[0].plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated')
    axes[0].set_xlabel('Mean Predicted Probability', fontsize=12)
    axes[0].set_ylabel('Fraction of Positives', fontsize=12)
    axes[0].set_title('🔬 Calibration Curves (Per Class)', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Histogram of predicted probabilities
    for i in range(num_classes):
        cls_name = class_names_list[i] if i < len(class_names_list) else f'Class {i}'
        axes[1].hist(y_pred_proba[:, i], bins=20, alpha=0.5, label=cls_name)
    axes[1].set_xlabel('Predicted Probability', fontsize=12)
    axes[1].set_ylabel('Count', fontsize=12)
    axes[1].set_title('🔬 Distribution of Predicted Probabilities', fontsize=14, fontweight='bold')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('calibration_curves.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ Run predictions first!")

## 37. Brain Region Annotation Overlay

Overlay anatomical annotations (hippocampus, ventricles) on SHAP heatmaps.

In [ ]:
# Brain Region Annotation Overlay
# Approximate brain region locations for 128x128 axial MRI slice
brain_regions = {
    'Hippocampus (L)': {'center': (45, 85), 'color': 'cyan'},
    'Hippocampus (R)': {'center': (83, 85), 'color': 'cyan'},
    'Ventricles': {'center': (64, 60), 'color': 'yellow'},
    'Frontal Cortex': {'center': (64, 30), 'color': 'lime'},
    'Temporal Lobe (L)': {'center': (25, 70), 'color': 'orange'},
    'Temporal Lobe (R)': {'center': (103, 70), 'color': 'orange'},
}

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for idx, (ax, i) in enumerate(zip(axes.flat, [0, 1, 2, 3])):
    img = test_subset[i]
    pred_cls = np.argmax(pred_subset[i])
    shap_2d = shap_values[pred_cls][i].squeeze()
    
    ax.imshow(img.squeeze(), cmap='gray', alpha=0.6)
    ax.imshow(shap_2d, cmap='RdBu_r', alpha=0.5, vmin=-np.abs(shap_2d).max(), vmax=np.abs(shap_2d).max())
    
    # Add region annotations
    for region, info in brain_regions.items():
        cx, cy = info['center']
        color = info['color']
        circle = plt.Circle((cx, cy), 8, fill=False, color=color, linewidth=2, linestyle='--')
        ax.add_patch(circle)
        ax.annotate(region, xy=(cx, cy), xytext=(cx+15, cy-10), fontsize=7, color=color,
                    arrowprops=dict(arrowstyle='->', color=color, lw=0.5))
    
    ax.set_title(f'Sample {i}: {pred_labels[i]}', fontsize=12, fontweight='bold')
    ax.axis('off')

fig.suptitle('Brain Region Annotations with SHAP Overlay', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('brain_region_annotations.png', dpi=300, bbox_inches='tight')
plt.show()

## 38. Correlation Heatmap – Feature Relationships

Correlations between top embedding features and predictions.

In [ ]:
# Correlation Heatmap - Feature Relationships
# Select top 20 embedding features by variance
feature_var = embeddings_test.var(axis=0)
top_feat_idx = np.argsort(feature_var)[-20:]
top_feats = embeddings_test[:, top_feat_idx]

df_feats = pd.DataFrame(top_feats, columns=[f'F{i}' for i in top_feat_idx])
df_feats['Pred_Prob'] = max_probs
df_feats['Class_Idx'] = le.transform(y_test_labels)

# Correlation matrix
corr = df_feats.corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0, ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=300)
plt.show()

## 39. Box Plot – SHAP Value Distribution by Class

Distribution of SHAP values across different dementia stages.

In [ ]:
# Box Plot - SHAP Value Distribution by Class
shap_data = []
for i, cls in enumerate(le.classes_):
    mean_shap_per_sample = np.mean(np.abs(shap_values[i]), axis=(1, 2, 3))
    for val in mean_shap_per_sample:
        shap_data.append({'Class': cls, 'Mean |SHAP|': val})

df_shap = pd.DataFrame(shap_data)

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df_shap, x='Class', y='Mean |SHAP|', palette='viridis', ax=ax)
sns.stripplot(data=df_shap, x='Class', y='Mean |SHAP|', color='black', alpha=0.3, size=4, ax=ax)
ax.set_title('SHAP Value Distribution by Predicted Class', fontsize=14, fontweight='bold')
ax.set_xlabel('Dementia Stage', fontsize=12)
ax.set_ylabel('Mean |SHAP| Value', fontsize=12)
plt.tight_layout()
plt.savefig('shap_boxplot.png', dpi=300)
plt.show()

## 40. Treemap – Misclassification Analysis

Hierarchical breakdown of correct and incorrect predictions.

In [ ]:
# Treemap - Misclassification Analysis
df_tree = df_sunburst.groupby(['True', 'Predicted', 'Result']).count().reset_index()

fig = px.treemap(df_tree, path=['Result', 'True', 'Predicted'], values='Count',
                 color='Result', color_discrete_map={'Correct': '#4CAF50', 'Incorrect': '#F44336'},
                 title='Treemap: Prediction Results by Class')
fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))
fig.write_html('treemap_predictions.html')
fig.show()

## 41. Plotly Interactive Clinical Dashboard

Fully interactive dashboard with all explainability components.

In [ ]:
# Plotly Interactive Clinical Dashboard
from plotly.subplots import make_subplots

def create_interactive_dashboard(idx=0):
    img = test_subset[idx]
    pred = pred_subset[idx]
    pred_cls = np.argmax(pred)
    true_cls_name = true_labels[idx]
    pred_cls_name = pred_labels[idx]
    shap_2d = shap_values[pred_cls][idx].squeeze()
    heatmap = get_grad_cam(model, img, pred_cls)
    
    fig = make_subplots(rows=2, cols=3,
                        subplot_titles=('Original MRI', 'Grad-CAM', 'SHAP Heatmap',
                                       'Prediction Probabilities', 'Embedding (UMAP)', 'Summary'),
                        specs=[[{'type': 'heatmap'}, {'type': 'heatmap'}, {'type': 'heatmap'}],
                               [{'type': 'bar'}, {'type': 'scatter'}, {'type': 'table'}]])
    
    # Original
    fig.add_trace(go.Heatmap(z=img.squeeze()[::-1], colorscale='Gray', showscale=False), row=1, col=1)
    
    # Grad-CAM
    fig.add_trace(go.Heatmap(z=heatmap[::-1], colorscale='Jet', showscale=False), row=1, col=2)
    
    # SHAP
    fig.add_trace(go.Heatmap(z=shap_2d[::-1], colorscale='RdBu', zmid=0, showscale=True), row=1, col=3)
    
    # Probabilities
    fig.add_trace(go.Bar(x=le.classes_, y=pred, marker_color=['#4CAF50' if i == pred_cls else '#2196F3' for i in range(num_classes)]), row=2, col=1)
    
    # UMAP point
    fig.add_trace(go.Scatter(x=umap_emb_2d[:, 0], y=umap_emb_2d[:, 1], mode='markers',
                             marker=dict(color=[1 if i == idx else 0 for i in range(len(umap_emb_2d))],
                                        colorscale='Reds', size=8)), row=2, col=2)
    
    # Summary table
    fig.add_trace(go.Table(header=dict(values=['Metric', 'Value']),
                           cells=dict(values=[['True Class', 'Predicted Class', 'Confidence', 'Correct'],
                                             [true_cls_name, pred_cls_name, f'{pred[pred_cls]:.2%}', str(true_cls_name == pred_cls_name)]])), row=2, col=3)
    
    fig.update_layout(height=800, title_text=f'NeuroXAI Interactive Dashboard - Sample {idx}', showlegend=False)
    return fig

fig = create_interactive_dashboard(0)
fig.write_html('interactive_dashboard.html')
fig.show()

## 42. Model Export – TensorFlow Lite Conversion

Save model and convert to TFLite for mobile deployment.

In [ ]:
# ====== MODEL EXPORT - TENSORFLOW LITE CONVERSION ======
if DATA_LOADED and 'model' in dir():
    # Save Keras model
    model.save('alzheimer_cnn_final.keras')
    print('✅ Keras model saved: alzheimer_cnn_final.keras')

    # Convert to TFLite
    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        tflite_model = converter.convert()
        with open('alzheimer_model.tflite', 'wb') as f:
            f.write(tflite_model)
        print(f'✅ TFLite model saved: alzheimer_model.tflite ({len(tflite_model)/1024:.1f} KB)')
    except Exception as e:
        print(f'⚠️ TFLite conversion failed: {e}')

    # Save class labels
    import json
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    with open('class_labels.json', 'w') as f:
        json.dump({'classes': class_names_list, 'class_mapping': CLASS_LABELS}, f, indent=2)
    print('✅ Class labels saved: class_labels.json')
else:
    print("⚠️ Train model first!")

## 43. Comprehensive Model Summary Report

Generate final summary of all metrics and outputs.

In [ ]:
# ====== COMPREHENSIVE MODEL SUMMARY REPORT ======
if DATA_LOADED and 'model' in dir() and 'y_pred' in dir():
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef
    
    class_names_list = [CLASS_LABELS.get(i, f'Class {i}') for i in range(num_classes)]
    
    summary = {
        'Dataset': {
            'Training Samples': len(X_train),
            'Validation Samples': len(X_val),
            'Test Samples': len(X_test),
            'Classes': class_names_list,
            'Image Size': f'{IMG_SIZE[0]}x{IMG_SIZE[1]}',
            'Data Source': 'Parquet files (MRI Dataset)'
        },
        'Model': {
            'Architecture': 'NeuroXAI Custom CNN (4 Conv Blocks)',
            'Total Parameters': f"{model.count_params():,}",
            'Trainable Parameters': f"{sum([np.prod(v.shape) for v in model.trainable_weights]):,}"
        },
        'Performance': {
            'Accuracy': accuracy_score(y_true, y_pred),
            'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred),
            'Cohen Kappa': cohen_kappa_score(y_true, y_pred),
            'Matthews Corr': matthews_corrcoef(y_true, y_pred),
            'Macro AUC-ROC': roc_auc['macro'] if 'roc_auc' in dir() else 'N/A',
            'Micro AUC-ROC': roc_auc['micro'] if 'roc_auc' in dir() else 'N/A'
        },
        'Per-Class AUC': {cls_name: roc_auc[i] for i, cls_name in enumerate(class_names_list)} if 'roc_auc' in dir() else {}
    }

    print('=' * 70)
    print('🧠 NEUROXAI - ALZHEIMER\'S DETECTION MODEL SUMMARY REPORT')
    print('=' * 70)
    
    for section, data in summary.items():
        print(f'\n📋 {section}:')
        print('-' * 50)
        if isinstance(data, dict):
            for k, v in data.items():
                if isinstance(v, float):
                    print(f'   {k}: {v:.4f}')
                else:
                    print(f'   {k}: {v}')
    
    print('\n' + '=' * 70)
    print('📁 Files Generated:')
    print('-' * 50)
    
    import glob
    output_files = glob.glob('*.png') + glob.glob('*.html') + glob.glob('*.keras') + glob.glob('*.tflite') + glob.glob('*.json')
    for f in sorted(output_files):
        print(f'   ✓ {f}')
    
    print('=' * 70)
    print('\n🎉 NeuroXAI Analysis Complete!')
    print('   📊 All visualizations saved as HD PNG files')
    print('   🌐 Interactive dashboards saved as HTML')
    print('   💾 Model exported to Keras and TFLite formats')
    print('   🔬 Explainability: SHAP, Grad-CAM, Integrated Gradients')
else:
    print("⚠️ Complete the full pipeline to generate summary!")